# 🧭 RFM Customer Segmentation
**L0 data health check · L1 customer segmentation** — built on the standard RFM field spec (`RFM_Fields_v2.xlsx`).

### What this notebook gives you
| | |
|---|---|
| ✅ **Data health check (L0)** | Whether the client's export is fit for RFM, what was cleaned automatically, and what the client must fix |
| 👥 **Customer segments (L1)** | Every customer scored 1–5 on Recency, Frequency and Monetary, and placed in one actionable segment |
| 📊 **KPIs and charts** | Segment size vs revenue, R × F heatmap, revenue concentration, distributions, segment map |
| 💡 **Generated insights** | Plain-language findings you can paste into a client deck |
| 📁 **Exports** | Per-customer segments, KPIs, data-quality report, thresholds (CSV, optional BigQuery, optional ZIP download) |

### How to run — 3 steps
1. Fill in **⚙️ 1a · Main settings** below (where the data is, client name, snapshot date, currency). 1b and 1c are optional.
2. Choose **Runtime → Run all** (Ctrl/⌘ + F9). Approve the Google Drive / Google account prompt if it appears.
3. Read the results from top to bottom. If the data has a blocking problem, the run stops at a **red box** that says what to fix.

> 💡 The code is hidden — you only work with the form fields. To peek at the code, double-click a cell's title.

### Input tables
| Table | Used for | Status |
|---|---|---|
| `transactions` | Everything — one row per purchase or return | **Required** |
| `customers` | Linking email ↔ customer ID, *New* vs *Reactivated*, loyalty and consent shares | Recommended |
| `fx_rates` | Converting all amounts to one currency (`month`, `currency`, `rate`) | Required if more than one currency |
| `items` | Checking that item lines add up to the transaction amount | Optional |

Column names should follow the spec. If the client uses other names, map them in **1b · Advanced settings**.

# 1 · Settings

In [2]:
# @title ⚙️ 1a · Main settings { display-mode: "form" }
# @markdown Fill in these fields, then choose **Runtime → Run all**.

# @markdown ---
# @markdown ### 📂 Where is the data?
SOURCE = "Google Drive folder"  # @param ["Google Drive folder", "Upload files", "BigQuery", "Local folder"]
# @markdown **Folder** (for Google Drive or Local) — after Drive is connected, *My Drive* is `/content/drive/MyDrive`. Tip: in the 📁 sidebar, right-click a folder → *Copy path*.
DATA_FOLDER = "/content/drive/MyDrive/RFMDummyData"  # @param {type:"string"}
# @markdown **BigQuery** — the project that pays for the queries, and the dataset with the tables (`project.dataset`).
BQ_BILLING_PROJECT = "your-gcp-project"  # @param {type:"string"}
BQ_DATASET = "your-gcp-project.rfm_dummy"  # @param {type:"string"}

# @markdown ---
# @markdown ### 📄 Tables from the client
# @markdown File names for Drive / Upload / Local. For BigQuery the table name is the same without `.csv`. **Leave a field empty** if the client did not provide that table.
TRANSACTIONS_FILE = "transactions.csv"  # @param {type:"string"}
CUSTOMERS_FILE = "customers.csv"  # @param {type:"string"}
FX_RATES_FILE = "fx_rates.csv"  # @param {type:"string"}
ITEMS_FILE = "items.csv"  # @param {type:"string"}

# @markdown ---
# @markdown ### 🗓️ Analysis
CLIENT_NAME = "Dummy Nordic Retail"  # @param {type:"string"}
# @markdown **Snapshot date** — recency is counted up to this day (the day itself is not included). Tick the box below to use the day after the latest transaction instead.
SNAPSHOT_DATE = "2026-11-01"  # @param {type:"date"}
USE_DAY_AFTER_LATEST_TRANSACTION = False  # @param {type:"boolean"}
# @markdown **Lookback window** (months) — 12 for fast-moving categories, 24 as default, 36 for long purchase cycles (furniture, optics).
LOOKBACK_MONTHS = 24  # @param {type:"slider", min:6, max:48, step:6}
# @markdown **Reporting currency** — all amounts are converted to this currency.
REPORTING_CURRENCY = "SEK"  # @param ["SEK", "NOK", "DKK", "EUR", "USD", "GBP"] {allow-input: true}

# @markdown ---
# @markdown ### 💾 Output
# @markdown Results are saved to `<data folder>/output/rfm_<snapshot date>` (change it in 1b).
DOWNLOAD_RESULTS_AS_ZIP = False  # @param {type:"boolean"}
SAVE_CHARTS_AS_HTML = True  # @param {type:"boolean"}

In [3]:
# @title 🛠️ 1b · Advanced settings (optional) { display-mode: "form" }
# @markdown The defaults work for most clients. Keep these unchanged between recurring runs of the same client, so results stay comparable.

# @markdown ---
# @markdown ### 🔤 Column renames
# @markdown Use when the client's column names differ from the spec. Format: `client_column=spec_column`, separated by commas (not case-sensitive).
# @markdown Example: `OrderID=transaction_id, OrderDate=transaction_date, Total=transaction_amount`
RENAME_TRANSACTIONS = ""  # @param {type:"string"}
RENAME_CUSTOMERS = ""  # @param {type:"string"}
RENAME_ITEMS = ""  # @param {type:"string"}
RENAME_FX_RATES = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎯 Scoring
# @markdown **Frequency bins** — lowest purchase count for scores 1 to 5 (five ascending numbers, starting at 1). Default `1, 2, 3, 4, 6` means 1 · 2 · 3 · 4–5 · 6+.
F_BINS = "1, 2, 3, 4, 6"  # @param {type:"string"}
# @markdown **Frozen thresholds** — path to the `rfm_thresholds.json` of an earlier run, to score with the same cut-offs and track how customers move between segments. Leave empty to calculate fresh cut-offs.
FROZEN_THRESHOLDS_PATH = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🚩 Outliers and B2B
# @markdown Customers far above this quantile in purchases or revenue are flagged — usually B2B, staff or test accounts.
OUTLIER_QUANTILE = 0.95  # @param {type:"number"}
OUTLIER_HANDLING = "Exclude from scoring, report separately"  # @param ["Exclude from scoring, report separately", "Keep in scoring"]
# @markdown **Items check tolerance** (%) — allowed difference between a transaction amount and the sum of its item lines.
RECONCILIATION_TOLERANCE_PCT = 1  # @param {type:"slider", min:0, max:10, step:0.5}

# @markdown ---
# @markdown ### 📤 Output destination
# @markdown Leave the folder empty for the default location.
OUTPUT_FOLDER = ""  # @param {type:"string"}
WRITE_TO_BIGQUERY = False  # @param {type:"boolean"}
BQ_OUTPUT_TABLE = "your-gcp-project.rfm_output.customer_segments"  # @param {type:"string"}

In [4]:
# @title 🧩 1c · Segment rules (optional) { display-mode: "form" }
# @markdown Each line is `Segment name | condition`. Rules are checked **top to bottom and the first match wins** — keep specific segments first and a catch-all last. Leave a line empty to skip it.
# @markdown
# @markdown You can use: `R`, `F`, `M` (scores 1–5) · `recency_days` · `frequency` (number of purchases) · `monetary` (net revenue) · `has_prior_history` (bought before the window; needs `first_purchase_date`). Operators: `>=`, `<=`, `==`, `and`, `or`, `not`.
RULE_01 = "Champions | R >= 4 and F >= 4 and M >= 4"  # @param {type:"string"}
RULE_02 = "Loyal High Value | R >= 3 and F >= 4 and M >= 4"  # @param {type:"string"}
RULE_03 = "At Risk High Value | R <= 2 and F >= 3 and M >= 4"  # @param {type:"string"}
RULE_04 = "New Customers | R >= 4 and F == 1 and not has_prior_history"  # @param {type:"string"}
RULE_05 = "Reactivated | R >= 4 and F == 1 and has_prior_history"  # @param {type:"string"}
RULE_06 = "Potential Loyalists | R >= 4"  # @param {type:"string"}
RULE_07 = "Needs Attention | R == 3"  # @param {type:"string"}
RULE_08 = "Hibernating | R <= 2"  # @param {type:"string"}
RULE_09 = ""  # @param {type:"string"}
RULE_10 = ""  # @param {type:"string"}
# @markdown ---
# @markdown 💡 To drop *Reactivated*, clear rule 05 — those customers then fall into *Potential Loyalists*.

In [5]:
# @title 🔧 Prepare the notebook and check the settings
# @markdown Loads the libraries, sets up the visual style and checks every setting above. If something is off, you get a red box with the fix — before any data is read.
import sys, os, re, io, json, zipfile, html as _html
from datetime import datetime
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from IPython import get_ipython

for _k in [k for k in list(globals()) if k.startswith("DONE_")]:
    del globals()[_k]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pio.renderers.default = "colab"

# ---------------- Chart style ----------------
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "text": "#0b0b0b", "muted": "#52514e",
     "grid": "#e6e5e1", "surface": "#fcfcfb"}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
FIGS = {}

def style(fig, title, height=420):
    fig.update_layout(title=dict(text=title, x=0, font=dict(size=16, color=C["text"])),
                      font=dict(family="Arial", size=12, color=C["text"]),
                      paper_bgcolor=C["surface"], plot_bgcolor=C["surface"], height=height,
                      margin=dict(l=60, r=30, t=70, b=50), hoverlabel=dict(bgcolor="white"),
                      legend=dict(orientation="h", y=1.08, x=0))
    fig.update_xaxes(gridcolor=C["grid"], zeroline=False, linecolor=C["grid"])
    fig.update_yaxes(gridcolor=C["grid"], zeroline=False, linecolor=C["grid"])
    return fig

def show_fig(fig, key, title, height=420):
    style(fig, title, height)
    FIGS[key] = fig
    fig.show()

# ---------------- UI helpers ----------------
CSS = """<style>
.rfm{font-family:'Google Sans',Inter,Arial,sans-serif;color:#0b0b0b;font-size:14px;line-height:1.5;max-width:1150px}
.rfm *{box-sizing:border-box}
.rfm .card{background:#fff;border:1px solid #e6e5e1;border-left-width:4px;border-radius:10px;padding:12px 16px;margin:8px 0}
.rfm .card.ok{border-left-color:#1baf7a}.rfm .card.warn{border-left-color:#e8a317;background:#fffdf6}
.rfm .card.err{border-left-color:#d93025;background:#fff8f7}.rfm .card.info{border-left-color:#2a78d6}
.rfm .card h4{margin:0 0 4px;font-size:15px;color:#0b0b0b}
.rfm .card ul{margin:4px 0 0;padding-left:20px}.rfm .card li{margin:3px 0}
.rfm .muted{color:#52514e;font-size:12.5px}
.rfm .kpis{display:flex;flex-wrap:wrap;gap:10px;margin:8px 0}
.rfm .kpi{flex:1 1 170px;background:#fff;border:1px solid #e6e5e1;border-radius:10px;padding:12px 14px}
.rfm .kpi .l{font-size:11px;color:#52514e;text-transform:uppercase;letter-spacing:.05em}
.rfm .kpi .v{font-size:22px;font-weight:700;margin-top:2px;color:#0b0b0b}
.rfm .kpi .s{font-size:12px;color:#52514e}
.rfm table.t{border-collapse:collapse;width:100%;font-size:13px;background:#fff;margin:6px 0;color:#0b0b0b}
.rfm table.t th{text-align:left;font-weight:600;color:#52514e;border-bottom:2px solid #e6e5e1;padding:7px 10px;font-size:11.5px;text-transform:uppercase;letter-spacing:.03em;white-space:nowrap}
.rfm table.t td{border-bottom:1px solid #f0efeb;padding:6px 10px;vertical-align:top}
.rfm table.t td.n{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
.rfm table.t tr.total td{font-weight:700;border-top:2px solid #e6e5e1;background:#fcfcfb}
.rfm .b{display:inline-block;padding:1px 9px;border-radius:999px;font-size:11px;font-weight:600;white-space:nowrap}
.rfm .b.critical{background:#fde2e1;color:#a50e0e}.rfm .b.warning{background:#fff4d6;color:#8a5a00}
.rfm .b.info{background:#e8f0fe;color:#174ea6}.rfm .b.ok{background:#e6f4ea;color:#137333}.rfm .b.na{background:#f1f3f4;color:#5f6368}
.rfm .bar{display:inline-block;height:9px;background:#9ec5f4;border-radius:4px;vertical-align:middle;margin-right:6px}
.rfm code{background:#f1f3f4;padding:1px 5px;border-radius:4px;font-size:12.5px;color:#0b0b0b}
.rfm h3{font-size:15px;margin:14px 0 4px;color:#0b0b0b;background:#fff;display:inline-block;padding:2px 8px;border-radius:6px}
</style>"""
esc = _html.escape

def show(*parts):
    display(HTML(CSS + '<div class="rfm">' + "".join(parts) + "</div>"))

def card(kind, title, body=""):
    return f'<div class="card {kind}"><h4>{title}</h4><div>{body}</div></div>'

def kpis(items):
    return '<div class="kpis">' + "".join(
        f'<div class="kpi"><div class="l">{esc(l)}</div><div class="v">{v}</div><div class="s">{s}</div></div>'
        for l, v, s in items) + "</div>"

def badge(kind, text=None):
    return f'<span class="b {kind}">{esc(text or kind)}</span>'

def bar(v, vmax, label):
    w = 0 if not vmax or pd.isna(v) else max(2, 90 * v / vmax)
    return f'<span class="bar" style="width:{w:.0f}px"></span>{label}'

def h3(text):
    return f"<h3>{text}</h3>"

def _isna(v):
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):
        return False

def table(df, fmt=None, html_cols=(), total_rows=()):
    fmt = fmt or {}
    head = "".join(f"<th>{esc(str(c))}</th>" for c in df.columns)
    body = []
    for i, (_, r) in enumerate(df.iterrows()):
        cells = []
        for c in df.columns:
            v = r[c]
            num = isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_))
            if c in html_cols:
                s = str(v)
            elif _isna(v):
                s = '<span class="muted">–</span>'
            elif c in fmt:
                s = esc(fmt[c](v))
            elif isinstance(v, (float, np.floating)):
                s = f"{v:,.2f}"
            elif num:
                s = f"{v:,}"
            else:
                s = esc(str(v))
            cells.append(f'<td class="n">{s}</td>' if num else f"<td>{s}</td>")
        cls = ' class="total"' if i in total_rows else ""
        body.append(f"<tr{cls}>{''.join(cells)}</tr>")
    return f'<div style="overflow-x:auto"><table class="t"><thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table></div>'

pct1 = lambda v: f"{v:.1%}"
pct0 = lambda v: f"{v:.0%}"
num0 = lambda v: f"{v:,.0f}"
num1 = lambda v: f"{v:,.1f}"

def money(v):
    a = abs(v)
    s = f"{v / 1e9:,.2f}B" if a >= 1e9 else f"{v / 1e6:,.1f}M" if a >= 1e6 else f"{v / 1e3:,.0f}k" if a >= 1e4 else f"{v:,.0f}"
    return f"{s} {REPORTING_CURRENCY}"

def takeaway(text):
    show(card("info", "📌 Takeaway", text))

# ---------------- Friendly stops (no tracebacks) ----------------
class RFMStop(Exception):
    pass

def stop(title, what, fix=""):
    e = RFMStop(title)
    e.html = card("err", "⛔ " + title, what + (f'<div style="margin-top:6px"><b>How to fix:</b> {fix}</div>' if fix else "")
                  + '<div class="muted" style="margin-top:8px">The run stopped here. After fixing, choose <b>Runtime → Run all</b> again.</div>')
    raise e

def _show_stop(shell, etype, evalue, tb, tb_offset=None):
    show(getattr(evalue, "html", card("err", "⛔ Stopped", esc(str(evalue)))))

if get_ipython() is not None:
    get_ipython().set_custom_exc((RFMStop,), _show_stop)

def need(*names, step="the steps above"):
    if any(n not in globals() for n in names):
        stop("An earlier step has not run", f"This step uses results from <b>{step}</b>.",
             "Choose <b>Runtime → Run all</b>, or run the cells above in order.")

# ---------------- Field spec ----------------
SPEC = {  # table -> column -> (type, requirement)
    "transactions": {
        "transaction_id": ("string", "Required"), "transaction_date": ("date", "Required"),
        "hashed_email": ("hash", "Required*"), "customer_id": ("string", "Required*"),
        "transaction_amount": ("number", "Required"), "transaction_type": ("string", "Recommended"),
        "original_transaction_id": ("string", "Optional"), "basket_size": ("number", "Optional"),
        "currency": ("string", "Required for L2 / multi-currency"), "market": ("string", "Required for L2"),
        "medium": ("string", "Required for L4"), "store_id": ("string", "Optional"),
    },
    "customers": {
        "hashed_email": ("hash", "Required*"), "customer_id": ("string", "Required*"),
        "first_purchase_date": ("date", "Recommended"), "loyalty_member": ("bool", "Optional"),
        "marketing_consent": ("bool", "Optional"),
    },
    "items": {
        "transaction_id": ("string", "Required"), "item_sku": ("string", "Required"),
        "item_quantity": ("number", "Required"), "item_price": ("number", "Required"),
        "item_category": ("string", "Required"), "item_sub_category": ("string", "Optional"),
        "item_discount": ("number", "Optional"),
    },
    "fx_rates": {"month": ("string", "Required"), "currency": ("string", "Required"), "rate": ("number", "Required")},
}

SEGMENT_INFO = {   # description and suggested action, written to the output
    "Champions":           ("Bought recently, often and spend the most.", "Reward, early access, referral; protect from discounting."),
    "Loyal High Value":    ("Frequent, high-spending customers, not quite as recent.", "Loyalty benefits, upsell premium ranges."),
    "At Risk High Value":  ("Used to buy often and spend a lot, but have not purchased for a long time.", "Win-back with personalised offers; highest revenue at stake."),
    "New Customers":       ("First-ever purchase, made recently.", "Onboarding flow, second-purchase incentive."),
    "Reactivated":         ("One recent purchase in the window, but with purchase history before it.", "Re-engage quickly to rebuild the habit."),
    "Potential Loyalists": ("Recent customers with a few purchases.", "Cross-sell, membership invitation."),
    "Needs Attention":     ("Average recency; starting to slip.", "Reminders, recommendations, limited-time offers."),
    "Hibernating":         ("Last purchase long ago, low frequency.", "Low-cost reactivation or suppress to save budget."),
}

# ---------------- Read and check the settings ----------------
need("SOURCE", step="⚙️ 1a · Main settings")
need("F_BINS", step="🛠️ 1b · Advanced settings")
need("RULE_01", step="🧩 1c · Segment rules")

SRC = {"Google Drive folder": "drive", "Upload files": "upload", "BigQuery": "bigquery", "Local folder": "local"}[SOURCE]
FILE_NAMES = {"transactions": TRANSACTIONS_FILE.strip() or None, "customers": CUSTOMERS_FILE.strip() or None,
              "fx_rates": FX_RATES_FILE.strip() or None, "items": ITEMS_FILE.strip() or None}
REPORTING_CURRENCY = REPORTING_CURRENCY.strip().upper()

if not FILE_NAMES["transactions"]:
    stop("Transactions table not set", "The transactions file / table name in 1a is empty — it is the only required table.",
         "Enter the file name (e.g. <code>transactions.csv</code>) in 1a.")
if SRC in ("drive", "upload") and not IN_COLAB:
    stop("This source needs Google Colab", f"<b>{esc(SOURCE)}</b> only works inside Colab.", "Choose <b>Local folder</b> in 1a.")
if SRC == "bigquery" and ("your-gcp-project" in BQ_BILLING_PROJECT or "your-gcp-project" in BQ_DATASET or "." not in BQ_DATASET):
    stop("BigQuery settings incomplete", "The billing project or dataset in 1a still has the placeholder value.",
         "Enter your GCP project and the dataset as <code>project.dataset</code>.")
if WRITE_TO_BIGQUERY and "your-gcp-project" in BQ_OUTPUT_TABLE:
    stop("BigQuery output table not set", "<b>Write to BigQuery</b> is ticked in 1b, but the output table is a placeholder.",
         "Enter <code>project.dataset.table</code>, or untick the box.")

def parse_renames(text, table_name):
    m = {}
    for part in [p.strip() for p in str(text).split(",") if p.strip()]:
        if "=" not in part:
            stop("Column rename not understood", f"In 1b ({table_name}): <code>{esc(part)}</code> has no <code>=</code>.",
                 "Use <code>client_column=spec_column</code>, separated by commas.")
        a, b = [x.strip().lower() for x in part.split("=", 1)]
        m[a] = b
    return m
COLUMN_MAP = {"transactions": parse_renames(RENAME_TRANSACTIONS, "transactions"),
              "customers": parse_renames(RENAME_CUSTOMERS, "customers"),
              "items": parse_renames(RENAME_ITEMS, "items"),
              "fx_rates": parse_renames(RENAME_FX_RATES, "fx_rates")}
unknown_targets = [f"{t}: {b}" for t, m in COLUMN_MAP.items() for b in m.values() if b not in SPEC[t]]

try:
    F_BIN_EDGES = [int(x) for x in re.split(r"[,\s;]+", F_BINS.strip()) if x]
except ValueError:
    F_BIN_EDGES = []
if len(F_BIN_EDGES) != 5 or F_BIN_EDGES[0] != 1 or F_BIN_EDGES != sorted(set(F_BIN_EDGES)):
    stop("Frequency bins not valid", f"1b has <code>{esc(F_BINS)}</code>.",
         "Enter five ascending whole numbers starting at 1, e.g. <code>1, 2, 3, 4, 6</code>.")
if not (0.9 <= OUTLIER_QUANTILE < 1):
    stop("Outlier quantile not valid", f"1b has {OUTLIER_QUANTILE}.", "Use a value between 0.9 and 0.9999, e.g. 0.999.")

SNAPSHOT_DATE_CFG = None if USE_DAY_AFTER_LATEST_TRANSACTION else (SNAPSHOT_DATE or None)
OUTLIER_ACTION = "exclude" if OUTLIER_HANDLING.startswith("Exclude") else "keep"
RECONCILIATION_TOLERANCE = RECONCILIATION_TOLERANCE_PCT / 100
FROZEN = FROZEN_THRESHOLDS_PATH.strip() or None

SEGMENT_RULES = []
for i in range(1, 11):
    v = str(globals().get(f"RULE_{i:02d}", "")).strip()
    if not v:
        continue
    if "|" not in v:
        stop("Segment rule not understood", f"Rule {i:02d} in 1c: <code>{esc(v)}</code> has no <code>|</code>.",
             "Write it as <code>Segment name | condition</code>.")
    name, rule = [x.strip() for x in v.split("|", 1)]
    if not name or not rule:
        stop("Segment rule incomplete", f"Rule {i:02d} in 1c needs both a name and a condition.", "Example: <code>Champions | R >= 4 and F >= 4</code>")
    SEGMENT_RULES.append((name, rule))
if not SEGMENT_RULES:
    stop("No segment rules", "All rules in 1c are empty.", "Add at least one rule, ideally ending with a catch-all.")
_d = pd.DataFrame(dict(R=[1, 3, 5], F=[1, 3, 5], M=[1, 3, 5], recency_days=[400, 90, 5], frequency=[1, 3, 8],
                       monetary=[100.0, 900.0, 9000.0], has_prior_history=[False, True, False]))
for name, rule in SEGMENT_RULES:
    try:
        res = _d.eval(rule)
        if not (isinstance(res, pd.Series) and res.dtype == bool):
            raise ValueError("the condition does not give True / False")
    except Exception as ex:
        stop("Segment rule not understood", f"<b>{esc(name)}</b>: <code>{esc(rule)}</code><br><span class='muted'>{esc(str(ex))}</span>",
             "Check the variable names (R, F, M, recency_days, frequency, monetary, has_prior_history) and use <code>and / or / not</code>.")

location = {"drive": DATA_FOLDER, "local": DATA_FOLDER, "upload": "files you upload in step 2", "bigquery": BQ_DATASET}[SRC]
settings_view = pd.DataFrame([
    ("Data source", f"{SOURCE} — {location}"),
    ("Tables", ", ".join(f"{t} ({fn})" for t, fn in FILE_NAMES.items() if fn)),
    ("Client", CLIENT_NAME),
    ("Snapshot date", "day after the latest transaction" if not SNAPSHOT_DATE_CFG else SNAPSHOT_DATE_CFG),
    ("Lookback window", f"{LOOKBACK_MONTHS} months"),
    ("Reporting currency", REPORTING_CURRENCY),
    ("Frequency bins", " · ".join(f"{a}" if b - a == 1 else f"{a}–{b - 1}" for a, b in zip(F_BIN_EDGES, F_BIN_EDGES[1:])) + f" · {F_BIN_EDGES[-1]}+"),
    ("R / M cut-offs", f"frozen from {FROZEN}" if FROZEN else "calculated from this data (quintiles)"),
    ("Outliers", f"flagged above the {OUTLIER_QUANTILE:.2%} quantile; {OUTLIER_HANDLING.lower()}"),
    ("Segment rules", f"{len(SEGMENT_RULES)} rules: " + " → ".join(n for n, _ in SEGMENT_RULES)),
], columns=["Setting", "Value"])
parts = [card("ok", "✅ Settings look good", "Here is what this run will use:"), table(settings_view)]
if unknown_targets:
    parts.append(card("warn", "⚠️ Some renames point to columns that are not in the spec",
                      ", ".join(f"<code>{esc(x)}</code>" for x in unknown_targets) + " — these columns will be ignored."))
show(*parts)
DONE_prep = True

Setting,Value
Data source,Google Drive folder — /content/drive/MyDrive/RFMDummyData
Tables,"transactions (transactions.csv), customers (customers.csv), fx_rates (fx_rates.csv), items (items.csv)"
Client,Dummy Nordic Retail
Snapshot date,2026-11-01
Lookback window,24 months
Reporting currency,SEK
Frequency bins,1 · 2 · 3 · 4–5 · 6+
R / M cut-offs,calculated from this data (quintiles)
Outliers,"flagged above the 95.00% quantile; exclude from scoring, report separately"
Segment rules,8 rules: Champions → Loyal High Value → At Risk High Value → New Customers → Reactivated → Potential Loyalists → Needs Attention → Hibernating


# 2 · Load the data
**What happens:** the notebook connects to the source you picked (mounts Google Drive, opens a file picker, or signs in to BigQuery) and reads each table.

**What you get:** a list of the tables found, with row and column counts. Missing optional tables are fine — the steps that need them are skipped.

In [6]:
# @title 📥 2 · Load the data
need("DONE_prep", step="🔧 Prepare the notebook")

def read_csv(path):
    # Everything as text first; types are parsed and validated in L0
    return pd.read_csv(path, dtype=str, keep_default_na=True)

def _finish(df, t):
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df.rename(columns=COLUMN_MAP.get(t, {}))

raw, notes, origin = {}, {}, {}
if SRC in ("drive", "local"):
    if SRC == "drive" and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    folder = DATA_FOLDER.strip()
    if not os.path.isdir(folder):
        stop("Data folder not found", f"<code>{esc(folder)}</code> does not exist.",
             "Check the folder path in 1a. In the 📁 sidebar, right-click the folder → <i>Copy path</i>.")
    present = sorted(os.listdir(folder))
    for t, fn in FILE_NAMES.items():
        if not fn:
            notes[t] = "not provided"; continue
        p = os.path.join(folder, fn)
        if os.path.exists(p):
            raw[t] = _finish(read_csv(p), t); origin[t] = fn
        else:
            notes[t] = f"file {fn} not found"
elif SRC == "upload":
    from google.colab import files
    wanted = [fn for fn in FILE_NAMES.values() if fn]
    show(card("info", "📤 Choose the files", "Click <b>Choose files</b> below and select: " + ", ".join(f"<code>{esc(f)}</code>" for f in wanted)))
    up = files.upload()
    lower = {k.lower(): k for k in up}
    present = list(up)
    for t, fn in FILE_NAMES.items():
        if not fn:
            notes[t] = "not provided"; continue
        k = lower.get(fn.lower())
        if k:
            raw[t] = _finish(pd.read_csv(io.BytesIO(up[k]), dtype=str), t); origin[t] = k
        else:
            notes[t] = "not uploaded"
elif SRC == "bigquery":
    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project=BQ_BILLING_PROJECT)
    present = []
    for t, fn in FILE_NAMES.items():
        if not fn:
            notes[t] = "not provided"; continue
        tid = f"{BQ_DATASET.strip()}.{re.sub(r'[.]csv$', '', fn.strip(), flags=re.I)}"
        try:
            df = client.query(f"SELECT * FROM `{tid}`").to_dataframe()
            raw[t] = _finish(df.astype("string").astype(object).where(df.notna(), None), t); origin[t] = tid
        except Exception as ex:
            notes[t] = f"could not read {tid} ({str(ex).splitlines()[0][:120]})"

if "transactions" not in raw:
    stop("Transactions table not found", f"{esc(notes.get('transactions', ''))}.<br>Found in the source: "
         + (", ".join(f"<code>{esc(p)}</code>" for p in present) or "nothing"),
         "Check the file / table name in 1a, or the folder / dataset.")

rows = []
for t in FILE_NAMES:
    if t in raw:
        rows.append(dict(Table=t, Source=origin[t], Rows=len(raw[t]), Columns=raw[t].shape[1], Status=badge("ok", "loaded")))
    else:
        rows.append(dict(Table=t, Source="–", Rows=np.nan, Columns=np.nan,
                         Status=badge("na" if notes[t] == "not provided" else "warning", notes[t])))
show(card("ok", f"✅ {len(raw)} of {sum(1 for f in FILE_NAMES.values() if f)} tables loaded", ""),
     table(pd.DataFrame(rows), fmt={"Rows": num0, "Columns": num0}, html_cols=("Status",)))
DONE_load = True

Mounted at /content/drive


Table,Source,Rows,Columns,Status
transactions,transactions.csv,"104,515",12,loaded
customers,customers.csv,"25,001",5,loaded
fx_rates,fx_rates.csv,125,3,loaded
items,items.csv,"159,784",7,loaded


# 3 · L0 — Data health check
Before any segmentation, the notebook checks whether the client's export can be trusted. Every issue is logged with a **severity**:

| Severity | Meaning |
|---|---|
| 🔴 **critical** | The client must fix the export. Results are only indicative until then. |
| 🟡 **warning** | Fixed automatically by the notebook (e.g. duplicates removed); worth mentioning to the client. |
| 🔵 **info** | For awareness — expected in most exports. |

The full list is in **3.6 L0 report**, and it is exported as `l0_dq_report.csv` — ready to send to the client.

### 3.1 Field mapping against the spec
**What you get:** every spec column with its status per table. 🔴 *missing* on a **Required** column stops the run; 🟡 *missing* on optional columns only limits later levels (e.g. no `market` → no L2 market split). `Required*` means at least one of `hashed_email` / `customer_id` is needed.

In [7]:
# @title 🗺️ 3.1 Field mapping
need("DONE_load", step="📥 2 · Load the data")

# fx_rates: accept any "rate*" column name (e.g. rate_to_sek)
if "fx_rates" in raw and "rate" not in raw["fx_rates"].columns:
    rc = [c for c in raw["fx_rates"].columns if c.startswith("rate")]
    if rc:
        raw["fx_rates"] = raw["fx_rates"].rename(columns={rc[0]: "rate"})

mapping_rows, extras = [], {}
for t, cols in SPEC.items():
    present = set(raw[t].columns) if t in raw else set()
    for col, (typ, req) in cols.items():
        if t not in raw:
            st = badge("na", "table not provided")
        elif col in present:
            st = badge("ok", "present")
        else:
            st = badge("critical" if req.startswith("Required") else "warning", "missing")
        mapping_rows.append(dict(Table=t, Column=col, Type=typ, Requirement=req, Status=st))
    if t in raw and present - set(cols):
        extras[t] = sorted(present - set(cols))
mapping = pd.DataFrame(mapping_rows)

tx_cols = set(raw["transactions"].columns)
critical_missing = [c for c in ["transaction_id", "transaction_date", "transaction_amount"] if c not in tx_cols]
if not ({"hashed_email", "customer_id"} & tx_cols):
    critical_missing.append("hashed_email or customer_id")

n_present = int((mapping.Status.str.contains(">present<")).sum())
parts = [kpis([("Spec columns found", f"{n_present} / {len(mapping)}", "across provided tables"),
               ("Extra columns ignored", f"{sum(len(v) for v in extras.values())}", "not in the spec")]),
         table(mapping, html_cols=("Status",))]
if extras:
    parts.append(card("info", "ℹ️ Columns not in the spec (ignored)",
                      "<br>".join(f"<b>{t}:</b> " + ", ".join(f"<code>{esc(c)}</code>" for c in cs) for t, cs in extras.items())
                      + "<div class='muted' style='margin-top:4px'>If one of these is a spec field under another name, map it in 1b · Column renames.</div>"))
show(*parts)
if critical_missing:
    stop("Required columns missing in transactions", ", ".join(f"<code>{esc(c)}</code>" for c in critical_missing),
         "Ask the client to add them, or — if they exist under another name — map them in 1b · Column renames.")
for c in SPEC["transactions"]:
    if c not in raw["transactions"].columns:
        raw["transactions"][c] = None   # optional columns added empty so the code below runs
DONE_map = True

Table,Column,Type,Requirement,Status
transactions,transaction_id,string,Required,present
transactions,transaction_date,date,Required,present
transactions,hashed_email,hash,Required*,present
transactions,customer_id,string,Required*,present
transactions,transaction_amount,number,Required,present
transactions,transaction_type,string,Recommended,present
transactions,original_transaction_id,string,Optional,present
transactions,basket_size,number,Optional,present
transactions,currency,string,Required for L2 / multi-currency,present
transactions,market,string,Required for L2,present


### 3.2 Cleaning: duplicates, dates, amounts, returns, currencies
**What happens:** duplicates are removed, dates and amounts are validated, transactions outside the lookback window are dropped, purchases vs returns are made consistent (returns always negative), hashed emails are checked (no plain emails, 64-char SHA-256), and all amounts are converted to the reporting currency with monthly FX rates.

**What you get:** the analysis window and a short summary. Details are in 3.6.

In [8]:
# @title 🧹 3.2 Clean the transactions
need("DONE_map", step="🗺️ 3.1 Field mapping")

dq, clog = [], []
def issue(table, check, severity, rows, total, action):
    dq.append(dict(table=table, check=check, severity=severity, rows=int(rows),
                   share=(rows / total if total else 0), action=action))
def log(step, before, after):
    clog.append(dict(step=step, rows_before=before, rows_after=after, removed=before - after))

def clean_hash(s, table):
    s = s.astype("string").str.strip()
    n = s.notna().sum()
    pii = s.str.contains("@", na=False)
    issue(table, "hashed_email contains plain email (PII)", "critical", pii.sum(), n, "Values removed; client must hash before sharing")
    upper = s.str.fullmatch(r"[0-9A-Fa-f]{64}", na=False) & ~s.str.fullmatch(r"[0-9a-f]{64}", na=False)
    issue(table, "hashed_email in uppercase", "warning", upper.sum(), n, "Lowercased")
    s = s.str.lower()
    bad = s.notna() & ~s.str.fullmatch(r"[0-9a-f]{64}", na=False)
    issue(table, "hashed_email not a 64-char SHA-256 hex", "warning", (bad & ~pii).sum(), n, "Set to empty; row falls back to customer_id")
    return s.where(~bad, pd.NA)

tx = raw["transactions"].copy()
N0 = len(tx)

# Duplicates
n = len(tx); dup = tx.duplicated()
issue("transactions", "exact duplicate rows", "warning", dup.sum(), n, "Removed")
tx = tx[~dup]; log("remove exact duplicates", n, len(tx))
n = len(tx); dup_id = tx.transaction_id.notna() & tx.transaction_id.duplicated(keep="first")
issue("transactions", "same transaction_id, different content", "warning", dup_id.sum(), n, "First row kept")
tx = tx[~dup_id]; log("remove conflicting duplicate ids", n, len(tx))
issue("transactions", "transaction_id empty", "warning", tx.transaction_id.isna().sum(), len(tx), "Rows removed")
n = len(tx); tx = tx[tx.transaction_id.notna()]; log("remove empty transaction_id", n, len(tx))

# Dates
tx["transaction_date"] = pd.to_datetime(tx.transaction_date.astype("string").str[:10], errors="coerce", format="%Y-%m-%d")
bad_date = tx.transaction_date.isna()
issue("transactions", "transaction_date missing or not YYYY-MM-DD", "warning" if bad_date.mean() < 0.05 else "critical",
      bad_date.sum(), len(tx), "Rows removed")
n = len(tx); tx = tx[~bad_date]; log("remove invalid dates", n, len(tx))
if tx.empty:
    stop("No valid dates", "No transaction_date could be read as <code>YYYY-MM-DD</code>.", "Ask the client for ISO dates (2026-01-31).")

SNAP = pd.Timestamp(SNAPSHOT_DATE_CFG) if SNAPSHOT_DATE_CFG else tx.transaction_date.max() + pd.Timedelta(days=1)
WSTART = SNAP - pd.DateOffset(months=LOOKBACK_MONTHS)
fut = tx.transaction_date >= SNAP
old = tx.transaction_date < WSTART
issue("transactions", f"date on/after snapshot {SNAP.date()}", "warning" if fut.mean() < 0.01 else "critical", fut.sum(), len(tx), "Rows removed")
issue("transactions", f"date before window start {WSTART.date()}", "info", old.sum(), len(tx), "Outside window; removed")
n = len(tx); tx = tx[~fut & ~old]; log("keep analysis window", n, len(tx))
if tx.empty:
    stop("No transactions in the analysis window", f"Window {WSTART.date()} → {(SNAP - pd.Timedelta(days=1)).date()} is empty.",
         "Check the snapshot date and lookback window in 1a.")

# Amounts
amt = pd.to_numeric(tx.transaction_amount.astype("string").str.replace(",", ".", regex=False).str.replace(r"\s", "", regex=True), errors="coerce")
issue("transactions", "transaction_amount missing or non-numeric", "warning", amt.isna().sum(), len(tx), "Rows removed")
tx["transaction_amount"] = amt
n = len(tx); tx = tx[amt.notna()]; log("remove invalid amounts", n, len(tx))

# Transaction type
tt = tx.transaction_type.astype("string").str.strip().str.lower()
tt = tt.replace({"refund": "return", "sale": "purchase", "order": "purchase"})
if tt.isna().all():
    tt = pd.Series(np.where(tx.transaction_amount < 0, "return", "purchase"), index=tx.index)
    issue("transactions", "transaction_type not provided", "info", len(tx), len(tx), "Inferred from sign of amount")
else:
    unk = ~tt.isin(["purchase", "return"])
    issue("transactions", "transaction_type not purchase/return", "warning", unk.sum(), len(tx), "Inferred from sign of amount")
    tt = tt.where(~unk, pd.Series(np.where(tx.transaction_amount < 0, "return", "purchase"), index=tx.index))
tx["transaction_type"] = tt
neg_p = (tx.transaction_type == "purchase") & (tx.transaction_amount < 0)
pos_r = (tx.transaction_type == "return") & (tx.transaction_amount > 0)
issue("transactions", "purchase with negative amount", "warning", neg_p.sum(), len(tx), "Treated as return")
issue("transactions", "return with positive amount", "warning", pos_r.sum(), len(tx), "Sign flipped to negative")
tx.loc[neg_p, "transaction_type"] = "return"
tx.loc[pos_r, "transaction_amount"] *= -1
zero = (tx.transaction_type == "purchase") & (tx.transaction_amount == 0)
issue("transactions", "purchase with zero amount", "info", zero.sum(), len(tx), "Kept (counts towards Frequency)")
ret = tx.transaction_type == "return"
issue("transactions", "return without original_transaction_id", "info",
      (ret & tx.original_transaction_id.isna()).sum(), ret.sum(), "Kept; reduces Monetary in its own month")

# Identifiers
tx["hashed_email"] = clean_hash(tx.hashed_email, "transactions")
tx["customer_id"] = tx.customer_id.astype("string").str.strip().replace("", pd.NA)

# Currency + FX
cur = tx.currency.astype("string").str.strip()
lower = cur.notna() & (cur != cur.str.upper())
issue("transactions", "currency not uppercase", "warning", lower.sum(), len(tx), "Uppercased")
cur = cur.str.upper()
miss = cur.isna()
if miss.any() and tx.market.notna().any():   # fill from the market's dominant currency
    mkt_cur = (pd.DataFrame({"m": tx.market, "c": cur}).dropna().groupby("m").c.agg(lambda s: s.mode().iat[0]))
    cur = cur.fillna(tx.market.map(mkt_cur))
issue("transactions", "currency missing", "warning", miss.sum(), len(tx),
      "Filled from the market's main currency" if tx.market.notna().any() else "Assumed reporting currency")
tx["currency"] = cur.fillna(REPORTING_CURRENCY)

currencies = sorted(tx.currency.unique())
if len(currencies) > 1 or currencies[0] != REPORTING_CURRENCY:
    if "fx_rates" not in raw:
        stop("FX rates needed", f"The data has currencies {', '.join(currencies)} but no <code>fx_rates</code> table was provided.",
             "Add an fx_rates table (<code>month, currency, rate</code> to " + REPORTING_CURRENCY + ") and set its name in 1a.")
    fx = raw["fx_rates"].copy()
    fx["currency"] = fx.currency.str.strip().str.upper()
    fx["rate"] = pd.to_numeric(fx.rate, errors="coerce")
    fx["month"] = fx.month.str[:7]
    fx = fx.dropna(subset=["month", "currency", "rate"]).drop_duplicates(["month", "currency"])
    tx["month"] = tx.transaction_date.dt.strftime("%Y-%m")
    tx = tx.merge(fx[["month", "currency", "rate"]], on=["month", "currency"], how="left")
    if tx.rate.isna().any():   # fallback: latest available month of the same currency
        last_rate = fx.sort_values("month").groupby("currency").rate.last()
        tx["rate"] = tx.rate.fillna(tx.currency.map(last_rate))
    tx.loc[tx.currency == REPORTING_CURRENCY, "rate"] = tx.loc[tx.currency == REPORTING_CURRENCY, "rate"].fillna(1.0)
    nofx = tx.rate.isna()
    issue("transactions", "no FX rate for currency", "critical" if nofx.mean() > 0.01 else "warning",
          nofx.sum(), len(tx), "Rows removed")
    n = len(tx); tx = tx[~nofx]; log("remove rows without FX rate", n, len(tx))
else:
    tx["rate"] = 1.0
tx["amount_rc"] = tx.transaction_amount * tx.rate     # amount in reporting currency

found = sum(1 for d in dq if d["rows"] > 0)
show(kpis([
    ("Analysis window", f"{LOOKBACK_MONTHS} months", f"{WSTART.date()} → {(SNAP - pd.Timedelta(days=1)).date()}"),
    ("Rows after cleaning", f"{len(tx):,}", f"of {N0:,} received ({len(tx) / N0:.0%})"),
    ("Currencies", ", ".join(currencies), f"converted to {REPORTING_CURRENCY}"),
    ("Checks with findings", f"{found}", "details in 3.6"),
]))
DONE_clean = True

### 3.3 Identity: one customer key per person
**What happens:** each transaction gets one `customer_key` — the hashed email where known; otherwise the email linked to the customer ID (via the customers table or rows that carry both); otherwise the customer ID itself. This merges a person's online and in-store purchases.

**What you get:** **coverage** — the share of purchases and revenue that can be tied to a known customer. Low retail coverage is normal (anonymous card payments), but it means in-store behaviour is under-represented in the segments.

In [9]:
# @title 🔗 3.3 Link email and customer ID
need("DONE_clean", step="🧹 3.2 Clean the transactions")

cu = raw.get("customers")
if cu is not None:
    cu = cu.copy()
    cu["hashed_email"] = clean_hash(cu["hashed_email"] if "hashed_email" in cu else pd.Series(index=cu.index, dtype="string"), "customers")
    cu["customer_id"] = (cu["customer_id"] if "customer_id" in cu else pd.Series(index=cu.index, dtype="string")).astype("string").str.strip().replace("", pd.NA)

pairs = [tx[["customer_id", "hashed_email"]]]
if cu is not None:
    pairs.append(cu[["customer_id", "hashed_email"]])
pairs = pd.concat(pairs).dropna()
cid2email = pairs.groupby("customer_id").hashed_email.agg(lambda s: s.value_counts().index[0]) if len(pairs) else pd.Series(dtype="string")
conflict = pairs.drop_duplicates().groupby("customer_id").hashed_email.nunique()
issue("identity", "customer_id linked to several emails", "info", (conflict > 1).sum(), len(conflict), "Most frequent email used")

linked = tx.hashed_email.isna() & tx.customer_id.map(cid2email).notna()
tx["customer_key"] = tx.hashed_email.fillna(tx.customer_id.map(cid2email))
tx["customer_key"] = tx.customer_key.fillna("cid:" + tx.customer_id)   # NA stays NA

ident = tx.customer_key.notna()
purch = tx.transaction_type == "purchase"
med = tx.medium.astype("string").str.lower()
has_medium = med.notna().any()
if has_medium:
    ecom_anon = (med == "ecommerce") & ~ident
    issue("transactions", "ecommerce transaction without any identifier", "warning", ecom_anon.sum(),
          (med == "ecommerce").sum(), "Excluded from RFM (check the export)")
issue("transactions", "transaction without identifier (anonymous)", "info", (~ident).sum(), len(tx), "Excluded from RFM")
issue("identity", "rows linked via customer_id → email", "info", linked.sum(), len(tx), "Merged into the email's customer")

coverage = {
    "Purchases identified": (ident & purch).sum() / max(purch.sum(), 1),
    "Revenue identified": tx.loc[ident, "amount_rc"].sum() / tx.amount_rc.sum() if tx.amount_rc.sum() else 0,
}
if has_medium:
    rp = purch & (med == "retail")
    if rp.any():
        coverage["Retail purchases linked"] = (rp & ident).sum() / rp.sum()
    ep = purch & (med == "ecommerce")
    if ep.any():
        coverage["Ecommerce purchases linked"] = (ep & ident).sum() / ep.sum()

n = len(tx); txi = tx[ident].copy(); log("keep identified transactions", n, len(txi))
if txi.empty:
    stop("No identified customers", "No transaction has a usable hashed_email or customer_id.",
         "Check the identifier columns in the export (see 3.6 for hash format issues).")

subs = {"Purchases identified": "of all purchases", "Revenue identified": "of net revenue",
        "Retail purchases linked": "in-store purchases with a customer", "Ecommerce purchases linked": "should be close to 100%"}
show(kpis([(k, f"{v:.0%}", subs.get(k, "")) for k, v in coverage.items()]
          + [("Rows linked via ID → email", f"{linked.sum():,}", "merged into one customer")]))
if "Ecommerce purchases linked" in coverage and coverage["Ecommerce purchases linked"] < 0.95:
    show(card("warn", "⚠️ Some online orders have no identifier", "Online orders should almost always carry an email or customer ID. Check the export with the client."))
DONE_ident = True

### 3.4 Items check
**What happens:** if the `items` table is provided, item lines are summed per transaction and compared with `transaction_amount`.

**What you get:** how many transactions don't add up. A systematic gap usually means a different basis (VAT, shipping, discounts) — fine for RFM, but it must be aligned before category analysis (L3).

In [10]:
# @title 🧾 3.4 Check items against transactions
need("DONE_ident", step="🔗 3.3 Link email and customer ID")

if "items" in raw and "item_price" in raw["items"].columns and "transaction_id" in raw["items"].columns:
    it = raw["items"].copy()
    it["item_price"] = pd.to_numeric(it.item_price, errors="coerce")
    orphan = ~it.transaction_id.isin(raw["transactions"].transaction_id)
    issue("items", "item line without matching transaction", "warning", orphan.sum(), len(it), "Ignored")
    issue("items", "item_price missing or non-numeric", "warning", it.item_price.isna().sum(), len(it), "Ignored in reconciliation")
    s = it.groupby("transaction_id").item_price.sum()
    rec = tx.set_index("transaction_id").transaction_amount.to_frame().join(s.rename("items_sum"), how="inner")
    diff = (rec.transaction_amount - rec.items_sum).abs() > RECONCILIATION_TOLERANCE * rec.transaction_amount.abs().clip(lower=1)
    issue("items", f"transaction_amount ≠ sum of item lines (>{RECONCILIATION_TOLERANCE:.0%})", "warning",
          diff.sum(), len(rec), "Transaction amount used for RFM; check VAT/shipping basis")
    no_items = ~tx.transaction_id.isin(it.transaction_id)
    issue("items", "transaction without item lines", "info", no_items.sum(), len(tx), "Relevant for L3")
    show(kpis([("Transactions matched to items", f"{len(rec):,}", f"{len(rec) / len(tx):.0%} of transactions"),
               ("Amounts that match", f"{1 - diff.mean():.0%}" if len(rec) else "–", f"within {RECONCILIATION_TOLERANCE:.0%}"),
               ("Item lines without a transaction", f"{orphan.sum():,}", "ignored")]))
    if len(rec) and diff.mean() > 0.05:
        show(card("warn", "⚠️ Item lines often don't add up", "Check with the client whether item prices include VAT, discounts or shipping in the same way as the transaction amount."))
else:
    show(card("info", "ℹ️ Items check skipped", "No items table (or no item_price column). This does not affect L1."))

### 3.5 Customer metrics and outliers
**What happens:** transactions are rolled up to one row per customer — last purchase date, number of purchases, gross revenue, returns and **net revenue** (Monetary). The customers table adds first purchase date, loyalty and consent. Customers far above the bulk (likely B2B, staff or test accounts) are flagged.

**What you get:** the list of flagged outliers. By default they are excluded from scoring, so they don't distort the cut-offs, and they are reported separately in the export.

In [11]:
# @title 👤 3.5 Build customer metrics and flag outliers
need("DONE_ident", step="🔗 3.3 Link email and customer ID")

p = txi[txi.transaction_type == "purchase"]
r = txi[txi.transaction_type == "return"]
cust = p.groupby("customer_key").agg(
    last_purchase=("transaction_date", "max"), first_purchase_window=("transaction_date", "min"),
    frequency=("transaction_id", "nunique"), gross_revenue=("amount_rc", "sum"))
if cust.empty:
    stop("No purchases from identified customers", "Only returns or anonymous purchases were found.", "Check transaction_type and identifiers in the export.")
cust["returns"] = r.groupby("customer_key").amount_rc.sum().reindex(cust.index).fillna(0)
cust["monetary"] = cust.gross_revenue + cust.returns
cust["aov"] = cust.gross_revenue / cust.frequency
cust["recency_days"] = (SNAP - cust.last_purchase).dt.days
only_ret = set(r.customer_key) - set(cust.index)
issue("customers", "customer with returns but no purchase in window", "info", len(only_ret), len(cust) + len(only_ret), "Excluded")
issue("customers", "net revenue ≤ 0 after returns (fully returned)", "info", int((cust.monetary <= 0).sum()), len(cust), "Kept; scored M = 1")

# Enrichment from the customers table
cust["first_purchase_date"] = pd.NaT
for c in ["loyalty_member", "marketing_consent"]:
    cust[c] = pd.NA
if cu is not None:
    cu["customer_key"] = cu.hashed_email.fillna(cu.customer_id.map(cid2email)).fillna("cid:" + cu.customer_id)
    cu = cu[cu.customer_key.notna()]
    dupk = cu.customer_key.duplicated()
    issue("customers", "duplicate customer rows", "info", dupk.sum(), len(cu), "Earliest first_purchase_date kept")
    def to_bool(s):
        return s.astype("string").str.strip().str.lower().map({"true": True, "1": True, "yes": True, "y": True,
                                                                 "false": False, "0": False, "no": False, "n": False})
    agg = {}
    if "first_purchase_date" in cu:
        cu["first_purchase_date"] = pd.to_datetime(cu.first_purchase_date.astype("string").str[:10], errors="coerce")
        agg["first_purchase_date"] = "min"
    for c in ["loyalty_member", "marketing_consent"]:
        if c in cu:
            cu[c] = to_bool(cu[c]); agg[c] = "max"
    if agg:
        enr = cu.groupby("customer_key").agg(agg)
        for c in enr.columns:
            cust[c] = enr[c].reindex(cust.index)
    matched = cust.index.isin(cu.customer_key)
    issue("customers", "RFM customers not found in customers table", "info", (~matched).sum(), len(cust),
          "No enrichment for these customers")
    later = cust.first_purchase_date > cust.first_purchase_window
    issue("customers", "first_purchase_date later than first purchase in data", "warning", later.sum(), len(cust),
          "Replaced by first purchase in data")
    cust.loc[later, "first_purchase_date"] = cust.loc[later, "first_purchase_window"]
cust["has_prior_history"] = (cust.first_purchase_date < cust.first_purchase_window).fillna(False).astype(bool)

# Outliers / B2B: only flag when far above the bulk (avoids flagging normal top customers in small datasets)
f_cut = max(cust.frequency.quantile(OUTLIER_QUANTILE), cust.frequency.quantile(0.99) * 3)
m_cut = max(cust.monetary.quantile(OUTLIER_QUANTILE), cust.monetary.quantile(0.99) * 3)
cust["is_outlier"] = (cust.frequency > f_cut) | (cust.monetary > m_cut)
issue("customers", f"outlier / possible B2B (F > {f_cut:,.0f} or M > {m_cut:,.0f} {REPORTING_CURRENCY})",
      "warning", cust.is_outlier.sum(), len(cust),
      "Excluded from scoring and reported separately" if OUTLIER_ACTION == "exclude" else "Kept")
outliers = cust[cust.is_outlier].sort_values("monetary", ascending=False)

parts = [kpis([("Customers", f"{len(cust):,}", "with a purchase in the window"),
               ("With history before the window", f"{cust.has_prior_history.mean():.0%}" if cu is not None else "–",
                "from first_purchase_date" if cu is not None else "no customers table"),
               ("Outliers flagged", f"{len(outliers):,}", f"{outliers.monetary.sum() / cust.monetary.sum():.1%} of net revenue" if len(outliers) else "none")])]
if len(outliers):
    ov = outliers.head(10).reset_index()[["customer_key", "frequency", "monetary", "aov", "recency_days"]]
    ov["customer_key"] = ov.customer_key.str[:16] + "…"
    ov.columns = ["Customer (shortened)", "Purchases", f"Net revenue ({REPORTING_CURRENCY})", "AOV", "Days since purchase"]
    parts += [h3("Top flagged customers"), table(ov, fmt={f"Net revenue ({REPORTING_CURRENCY})": num0, "AOV": num0})]
show(*parts)
DONE_cust = True

Customer (shortened),Purchases,Net revenue (SEK),AOV,Days since purchase
61a3a7f47234e637…,250,"6,229,171","24,917",1


### 3.6 L0 report
**What you get:** the overall verdict, every check with findings (sorted by severity) and the cleaning log — how many rows were removed at each step. This is the table to share with the client when the export needs fixing.

In [12]:
# @title 📋 3.6 L0 report
need("DONE_cust", step="👤 3.5 Build customer metrics")

dq_df = pd.DataFrame(dq)
dq_df = dq_df[dq_df.rows > 0].copy()
dq_df["_o"] = dq_df.severity.map({"critical": 0, "warning": 1, "info": 2})
dq_df = dq_df.sort_values(["_o", "rows"], ascending=[True, False]).drop(columns="_o")
clog_df = pd.DataFrame(clog)
n_crit, n_warn, n_info = [(dq_df.severity == s).sum() for s in ("critical", "warning", "info")]

verdict = (card("err", "🔴 Critical issues — results are indicative until the client fixes the export",
                "See the red rows below. Everything else ran, so you can still review the segments.") if n_crit else
           card("warn", "🟡 Data is usable — some issues were fixed automatically", "Review the warnings; mention the recurring ones to the client.") if n_warn else
           card("ok", "🟢 Data is clean", "No critical issues or warnings."))
view = dq_df.assign(severity=dq_df.severity.map(lambda s: badge(s)))[["severity", "table", "check", "rows", "share", "action"]]
view.columns = ["Severity", "Table", "Check", "Rows", "Share", "What the notebook did"]
cl = clog_df.copy(); cl.columns = ["Step", "Rows before", "Rows after", "Removed"]
show(verdict,
     kpis([("Rows received", f"{N0:,}", "transactions table"),
           ("Rows used", f"{len(txi):,}", f"{len(txi) / N0:.0%} — clean and identified"),
           ("Customers", f"{len(cust):,}", ""),
           ("Critical", f"{n_crit}", "must be fixed by the client"),
           ("Warnings", f"{n_warn}", "fixed automatically"),
           ("Info", f"{n_info}", "for awareness")]),
     h3("Checks with findings"),
     table(view, fmt={"Rows": num0, "Share": lambda v: f"{v:.2%}"}, html_cols=("Severity",)),
     h3("Cleaning log — transactions"),
     table(cl, fmt={c: num0 for c in ["Rows before", "Rows after", "Removed"]}))

Severity,Table,Check,Rows,Share,What the notebook did
warning,transactions,exact duplicate rows,40,0.04%,Removed
warning,transactions,ecommerce transaction without any identifier,27,0.05%,Excluded from RFM (check the export)
warning,items,transaction_amount ≠ sum of item lines (>1%),15,0.01%,Transaction amount used for RFM; check VAT/shipping basis
warning,transactions,currency not uppercase,8,0.01%,Uppercased
warning,transactions,hashed_email in uppercase,5,0.01%,Lowercased
warning,transactions,hashed_email not a 64-char SHA-256 hex,5,0.01%,Set to empty; row falls back to customer_id
warning,transactions,currency missing,4,0.00%,Filled from the market's main currency
warning,transactions,date on/after snapshot 2026-11-01,3,0.00%,Rows removed
warning,customers,"outlier / possible B2B (F > 48 or M > 91,553 SEK)",1,0.00%,Excluded from scoring and reported separately
info,transactions,transaction without identifier (anonymous),"17,781",17.02%,Excluded from RFM


# 4 · L1 — Customer segmentation
Every customer gets three scores from 1 (low) to 5 (high):

| Score | Measures | How it is scored |
|---|---|---|
| **R — Recency** | Days since the last purchase | Quintiles (5 = most recent 20%) |
| **F — Frequency** | Number of purchases in the window | Fixed bins (default 1 · 2 · 3 · 4–5 · 6+), because most customers buy once or twice |
| **M — Monetary** | Net revenue in the window, after returns | Quintiles (5 = top 20%) |

The scores are then combined into named segments with the rules from **1c**.

### 4.1 R, F and M scores
**What you get:** the cut-offs behind each score (e.g. *R = 5 means bought within the last N days*) and how many customers got each score. The cut-offs are saved as `rfm_thresholds.json` — reuse it in **1b · Frozen thresholds** next time, so a customer's movement between segments reflects behaviour, not shifting cut-offs.

In [13]:
# @title 🎯 4.1 Score customers
need("DONE_cust", step="👤 3.5 Build customer metrics")

scored = cust[~cust.is_outlier].copy() if OUTLIER_ACTION == "exclude" else cust.copy()
if FROZEN:
    if not os.path.exists(FROZEN):
        stop("Frozen thresholds file not found", f"<code>{esc(FROZEN)}</code>", "Check the path in 1b, or leave it empty to calculate new cut-offs.")
    with open(FROZEN) as f:
        TH = json.load(f)
    if TH.get("reporting_currency") != REPORTING_CURRENCY:
        show(card("warn", "⚠️ Currency differs from the frozen run", f"Frozen thresholds are in {TH.get('reporting_currency')}, this run uses {REPORTING_CURRENCY}."))
else:
    TH = {
        "recency_days_edges": scored.recency_days.quantile([0.2, 0.4, 0.6, 0.8]).round(0).tolist(),
        "monetary_edges": scored.monetary.quantile([0.2, 0.4, 0.6, 0.8]).round(2).tolist(),
        "frequency_bin_edges": F_BIN_EDGES,
        "reporting_currency": REPORTING_CURRENCY, "snapshot_date": str(SNAP.date()),
        "lookback_months": LOOKBACK_MONTHS,
    }
# R: fewer days = higher score; ties fall into the more recent bucket
scored["R"] = 5 - np.searchsorted(TH["recency_days_edges"], scored.recency_days, side="left")
scored["F"] = np.searchsorted(TH["frequency_bin_edges"], scored.frequency, side="right")
scored["M"] = 1 + np.searchsorted(TH["monetary_edges"], scored.monetary, side="right")
scored[["R", "F", "M"]] = scored[["R", "F", "M"]].clip(1, 5).astype(int)

re_, me, fe = TH["recency_days_edges"], TH["monetary_edges"], TH["frequency_bin_edges"]
f_labels = [f"{a}" if b - a == 1 else f"{a}–{b - 1}" for a, b in zip(fe, fe[1:])] + [f"{fe[-1]}+"]
th_table = pd.DataFrame({
    "Score": [5, 4, 3, 2, 1],
    "R — days since last purchase": [f"≤ {re_[0]:.0f}", f"{re_[0]:.0f}–{re_[1]:.0f}", f"{re_[1]:.0f}–{re_[2]:.0f}", f"{re_[2]:.0f}–{re_[3]:.0f}", f"> {re_[3]:.0f}"],
    "F — purchases": f_labels[::-1],
    f"M — net revenue ({REPORTING_CURRENCY})": [f"> {me[3]:,.0f}", f"{me[2]:,.0f}–{me[3]:,.0f}", f"{me[1]:,.0f}–{me[2]:,.0f}", f"{me[0]:,.0f}–{me[1]:,.0f}", f"≤ {me[0]:,.0f}"],
})
counts = pd.DataFrame({s: scored[s].value_counts().reindex(range(5, 0, -1)).fillna(0).astype(int) for s in "RFM"})
cnt = pd.DataFrame({"Score": counts.index, "Customers with R": counts.R.values, "Customers with F": counts.F.values, "Customers with M": counts.M.values})
show(card("info", "How to read", f"Customers scored: <b>{len(scored):,}</b>"
          + (f" ({len(outliers):,} outliers excluded)" if OUTLIER_ACTION == "exclude" and len(outliers) else "")
          + (". Cut-offs <b>frozen</b> from an earlier run." if FROZEN else ". Cut-offs calculated from this data.")
          + " R and M split customers into five equal groups; F uses the fixed bins, so its groups are uneven by design."),
     h3("Cut-offs per score"), table(th_table),
     h3("Customers per score"), table(cnt))
DONE_score = True

Score,R — days since last purchase,F — purchases,M — net revenue (SEK)
5,≤ 41,6+,"> 6,898"
4,41–89,4–5,"3,031–6,898"
3,89–310,3,"1,534–3,031"
2,310–504,2,"629–1,534"
1,> 504,1,≤ 629
Score,Customers with R,Customers with F,Customers with M
5,"5,022","4,217","5,000"
4,"5,017","2,504","4,999"
3,"4,984","2,846","5,000"
2,"4,998","7,040","4,999"


### 4.2 Segments
**What you get:** the segment rules in the order they are applied, with a description and a suggested action for each. Every customer lands in the **first** segment whose rule matches.

In [14]:
# @title 🧩 4.2 Assign segments
need("DONE_score", step="🎯 4.1 Score customers")

scored["segment"] = None
for name, rule in SEGMENT_RULES:
    free = scored.segment.isna()
    if not free.any():
        break
    hit = scored[free].eval(rule)
    scored.loc[hit[hit].index, "segment"] = name
unassigned = int(scored.segment.isna().sum())
if unassigned:
    scored["segment"] = scored.segment.fillna("Unassigned")
SEG_ORDER = [s for s, _ in SEGMENT_RULES if s in scored.segment.unique()] + (["Unassigned"] if unassigned else [])

seg_def = pd.DataFrame([dict(order=i + 1, segment=s, rule=rule, description=SEGMENT_INFO.get(s, ("", ""))[0],
                             suggested_action=SEGMENT_INFO.get(s, ("", ""))[1])
                        for i, (s, rule) in enumerate(SEGMENT_RULES)])
sizes = scored.segment.value_counts()
view = seg_def.assign(customers=seg_def.segment.map(sizes).fillna(0).astype(int))
view = view[["order", "segment", "rule", "customers", "description", "suggested_action"]]
view.columns = ["#", "Segment", "Rule", "Customers", "Description", "Suggested action"]
parts = [table(view, fmt={"Customers": num0})]
if unassigned:
    parts.insert(0, card("warn", f"⚠️ {unassigned:,} customers matched no rule", "They are shown as <b>Unassigned</b>. Add a catch-all rule at the end in 1c (e.g. <code>Other | R >= 1</code>)."))
empty = [s for s, _ in SEGMENT_RULES if s not in sizes.index]
if empty:
    parts.append(card("info", "ℹ️ Empty segments", ", ".join(esc(s) for s in empty) + " — no customers matched (earlier rules took them, or the data has none)."))
show(*parts)
DONE_seg = True

#,Segment,Rule,Customers,Description,Suggested action
1,Champions,R >= 4 and F >= 4 and M >= 4,"2,766","Bought recently, often and spend the most.","Reward, early access, referral; protect from discounting."
2,Loyal High Value,R >= 3 and F >= 4 and M >= 4,"1,682","Frequent, high-spending customers, not quite as recent.","Loyalty benefits, upsell premium ranges."
3,At Risk High Value,R <= 2 and F >= 3 and M >= 4,"1,780","Used to buy often and spend a lot, but have not purchased for a long time.",Win-back with personalised offers; highest revenue at stake.
4,New Customers,R >= 4 and F == 1 and not has_prior_history,"3,364","First-ever purchase, made recently.","Onboarding flow, second-purchase incentive."
5,Reactivated,R >= 4 and F == 1 and has_prior_history,737,"One recent purchase in the window, but with purchase history before it.",Re-engage quickly to rebuild the habit.
6,Potential Loyalists,R >= 4,"3,172",Recent customers with a few purchases.,"Cross-sell, membership invitation."
7,Needs Attention,R == 3,"3,302",Average recency; starting to slip.,"Reminders, recommendations, limited-time offers."
8,Hibernating,R <= 2,"8,195","Last purchase long ago, low frequency.",Low-cost reactivation or suppress to save budget.


### 4.3 Segment KPIs
**What you get:** headline numbers for the whole customer base, then one row per segment.

**How to read it:** compare **customer share** with **revenue share** — a segment with a much bigger revenue share than customer share is where value is concentrated. *Return rate* shows which segments return the most, and *consent share* shows how many you can actually reach with email, Customer Match or Meta audiences. All amounts are net of returns, in the reporting currency.

In [15]:
# @title 📈 4.3 Segment KPIs
need("DONE_seg", step="🧩 4.2 Assign segments")

tot_c, tot_m = len(scored), scored.monetary.sum()
g = scored.groupby("segment")
kpi = pd.DataFrame({
    "customers": g.size(),
    "customer_share": g.size() / tot_c,
    "net_revenue": g.monetary.sum(),
    "revenue_share": g.monetary.sum() / tot_m,
    "revenue_per_customer": g.monetary.mean(),
    "aov": g.gross_revenue.sum() / g.frequency.sum(),
    "avg_purchases": g.frequency.mean(),
    "median_days_since_purchase": g.recency_days.median(),
    "return_rate": -g.returns.sum() / g.gross_revenue.sum(),
}).reindex(SEG_ORDER)
if cust.loyalty_member.notna().any():
    kpi["loyalty_share"] = g.loyalty_member.apply(lambda s: s.fillna(False).astype(bool).mean()).reindex(SEG_ORDER)
if cust.marketing_consent.notna().any():
    kpi["consent_share"] = g.marketing_consent.apply(lambda s: s.fillna(False).astype(bool).mean()).reindex(SEG_ORDER)
total = pd.DataFrame({"customers": tot_c, "customer_share": 1.0, "net_revenue": tot_m, "revenue_share": 1.0,
                      "revenue_per_customer": scored.monetary.mean(),
                      "aov": scored.gross_revenue.sum() / scored.frequency.sum(),
                      "avg_purchases": scored.frequency.mean(),
                      "median_days_since_purchase": scored.recency_days.median(),
                      "return_rate": -scored.returns.sum() / scored.gross_revenue.sum()}, index=["Total"])
for c, src in [("loyalty_share", "loyalty_member"), ("consent_share", "marketing_consent")]:
    if c in kpi:
        total[c] = scored[src].fillna(False).astype(bool).mean()
kpi_all = pd.concat([kpi, total])
repeat = (scored.frequency >= 2).mean()

heads = [("Customers scored", f"{tot_c:,}", f"last {LOOKBACK_MONTHS} months"),
         ("Net revenue", money(tot_m), "after returns"),
         ("Revenue per customer", money(scored.monetary.mean()), f"{scored.frequency.mean():.1f} purchases on average"),
         ("Repeat rate", f"{repeat:.0%}", "bought 2+ times"),
         ("Return rate", f"{total.return_rate.iat[0]:.1%}", "of gross revenue")]
if "consent_share" in kpi:
    heads.append(("Marketing consent", f"{total.consent_share.iat[0]:.0%}", "addressable customers"))

LABELS = {"customers": "Customers", "customer_share": "Customer share", "net_revenue": f"Net revenue ({REPORTING_CURRENCY})",
          "revenue_share": "Revenue share", "revenue_per_customer": "Revenue / customer", "aov": "AOV",
          "avg_purchases": "Avg purchases", "median_days_since_purchase": "Median days since purchase",
          "return_rate": "Return rate", "loyalty_share": "Loyalty members", "consent_share": "Consent"}
view = kpi_all.copy()
vmax = kpi.revenue_share.max()
view["revenue_share"] = [bar(v, vmax if i != "Total" else None, f"{v:.1%}") for i, v in view.revenue_share.items()]
view = view.rename(columns=LABELS).rename_axis("Segment").reset_index()
fm = {"Customers": num0, "Customer share": pct1, LABELS["net_revenue"]: num0, "Revenue / customer": num0, "AOV": num0,
      "Avg purchases": num1, "Median days since purchase": num0, "Return rate": pct1, "Loyalty members": pct0, "Consent": pct0}
show(kpis(heads), table(view, fmt=fm, html_cols=("Revenue share",), total_rows=(len(view) - 1,)))
DONE_kpi = True

Segment,Customers,Customer share,Net revenue (SEK),Revenue share,Revenue / customer,AOV,Avg purchases,Median days since purchase,Return rate,Loyalty members,Consent
Champions,"2,766",11.1%,"38,613,162",33.5%,"13,960","1,656",8.8,33,3.7%,81%,60%
Loyal High Value,"1,682",6.7%,"13,439,059",11.7%,"7,990","1,590",5.2,168,4.2%,60%,57%
At Risk High Value,"1,780",7.1%,"26,968,260",23.4%,"15,151","1,921",8.2,461,4.1%,49%,60%
New Customers,"3,364",13.5%,"3,437,188",3.0%,"1,022","1,068",1.0,45,4.3%,49%,60%
Reactivated,737,2.9%,"987,300",0.9%,"1,340","1,418",1.0,47,5.5%,49%,62%
Potential Loyalists,"3,172",12.7%,"9,734,536",8.4%,"3,069","1,251",2.6,46,4.9%,52%,60%
Needs Attention,"3,302",13.2%,"9,399,897",8.1%,"2,847","1,178",2.6,214,5.9%,49%,60%
Hibernating,"8,195",32.8%,"12,773,107",11.1%,"1,559","1,100",1.5,517,5.7%,50%,60%
Total,"24,998",100.0%,"115,352,510",100.0%,"4,614","1,494",3.2,203,4.4%,54%,60%


### 4.4 Charts
All charts are interactive — hover for exact numbers, drag to zoom, double-click to reset. They are also saved as HTML files in the output folder (setting in 1a), ready to share.

#### 📊 Chart 1 — Share of customers vs share of revenue
**What it shows:** for each segment, its share of all customers (blue) next to its share of net revenue (orange).
**How to read it:** an orange bar much longer than the blue one marks a high-value segment — protect it. A long blue bar with a short orange one is a large, low-value group — reach it with cheap channels or suppress it.

In [16]:
# @title Chart 1 · Segment share of customers vs revenue
need("DONE_kpi", step="📈 4.3 Segment KPIs")
k = kpi.iloc[::-1]
fig = go.Figure()
fig.add_bar(y=k.index, x=k.customer_share, name="Share of customers", orientation="h", marker_color=C["blue"],
            text=[f"{v:.0%}" for v in k.customer_share], textposition="outside",
            hovertemplate="%{y}<br>Customers: %{x:.1%}<extra></extra>")
fig.add_bar(y=k.index, x=k.revenue_share, name="Share of revenue", orientation="h", marker_color=C["orange"],
            text=[f"{v:.0%}" for v in k.revenue_share], textposition="outside",
            hovertemplate="%{y}<br>Revenue: %{x:.1%}<extra></extra>")
fig.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08)
fig.update_xaxes(tickformat=".0%", range=[0, max(k.customer_share.max(), k.revenue_share.max()) * 1.15])
show_fig(fig, "1_segment_share", "Segments: share of customers vs share of revenue", height=460)

ratio = (kpi.revenue_share / kpi.customer_share).sort_values()
takeaway(f"<b>{esc(ratio.index[-1])}</b> is the most valuable segment per head: {kpi.at[ratio.index[-1], 'customer_share']:.0%} of customers "
         f"bring {kpi.at[ratio.index[-1], 'revenue_share']:.0%} of revenue. The largest segment is <b>{esc(kpi.customers.idxmax())}</b> "
         f"({kpi.customer_share.max():.0%} of customers, {kpi.at[kpi.customers.idxmax(), 'revenue_share']:.0%} of revenue).")

#### 🟦 Chart 2 — Customers by Recency and Frequency
**What it shows:** a grid of R score (left = long ago, right = recent) by F score (bottom = one purchase, top = many). Each cell holds the number of customers; hover to see their average revenue.
**How to read it:** the top-right corner is your loyal, active core. The bottom row is one-time buyers — usually the biggest group, and turning them into second-time buyers is often the cheapest growth lever. Top-left cells are valuable customers drifting away.

In [24]:
# @title Chart 2 · R × F heatmap
need("DONE_kpi", step="📈 4.3 Segment KPIs")

def _grid(values, agg):
    # Pivot F × R and force plain float64, so empty cells are NaN (not pd.NA, which Plotly cannot serialise)
    g = (scored.assign(R=scored.R.astype(int), F=scored.F.astype(int))
               .pivot_table(index="F", columns="R", values=values, aggfunc=agg)
               .reindex(index=range(5, 0, -1), columns=range(1, 6)))
    return g.apply(pd.to_numeric, errors="coerce").astype("float64")

hm_c = _grid("frequency", "size")   # customers per cell
hm_m = _grid("monetary", "mean")    # revenue per customer per cell
z = hm_c.to_numpy()
cd = hm_m.to_numpy()
labels = [["" if np.isnan(v) else f"{v:,.0f}" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=[f"R{c}" for c in hm_c.columns], y=[f"F{i}" for i in hm_c.index],
    colorscale=[[i / (len(SEQ_BLUE) - 1), c] for i, c in enumerate(SEQ_BLUE)],
    customdata=cd, xgap=2, ygap=2, colorbar=dict(title="Customers"),
    text=labels, texttemplate="%{text}", textfont=dict(size=11), hoverongaps=False,
    hovertemplate="%{x} · %{y}<br>Customers: %{z:,.0f}<br>Revenue / customer: %{customdata:,.0f} "
                  + REPORTING_CURRENCY + "<extra></extra>"))
fig.update_xaxes(title="Recency score (5 = most recent)", showgrid=False)
fig.update_yaxes(title="Frequency score", showgrid=False)
show_fig(fig, "2_rf_heatmap", "Customers by Recency and Frequency score", height=440)

stacked = hm_c.stack().dropna()
top_f, top_r = stacked.idxmax()
one_timers = (scored.F == 1).mean()
takeaway(f"The biggest cell is <b>R{top_r} · F{top_f}</b> with {stacked.max():,.0f} customers. "
         f"<b>{one_timers:.0%}</b> of customers are in the F1 row (one purchase in the window), and "
         f"{int(((scored.F == 1) & (scored.R >= 4)).sum()):,} of them bought recently — the best second-purchase audience.")

#### 📉 Chart 3 — Revenue concentration
**What it shows:** customers ranked from highest to lowest net revenue (x-axis) against the cumulative share of revenue they bring (y-axis).
**How to read it:** the steeper the curve, the more the business depends on a few customers. Read off the marked point: *the top 20% of customers bring X% of revenue*. Above ~70% means retention of the top tier matters more than acquisition.

In [19]:
# @title Chart 3 · Revenue concentration
need("DONE_kpi", step="📈 4.3 Segment KPIs")
srt = scored.monetary.clip(lower=0).sort_values(ascending=False).values
cum = np.cumsum(srt) / srt.sum()
x = np.arange(1, len(srt) + 1) / len(srt)
idx = np.linspace(0, len(srt) - 1, min(len(srt), 500)).astype(int)
top20 = cum[max(int(np.ceil(len(srt) * 0.2)) - 1, 0)]
half = (np.searchsorted(cum, 0.5) + 1) / len(srt)
fig = go.Figure(go.Scatter(x=x[idx], y=cum[idx], mode="lines", line=dict(color=C["blue"], width=2),
                           hovertemplate="Top %{x:.0%} of customers → %{y:.0%} of revenue<extra></extra>", name=""))
fig.add_trace(go.Scatter(x=[0.2], y=[top20], mode="markers+text", marker=dict(size=9, color=C["blue"], line=dict(color="white", width=2)),
                         text=[f"  Top 20% → {top20:.0%} of revenue"], textposition="middle right", hoverinfo="skip", showlegend=False))
fig.update_xaxes(title="Share of customers (ranked by net revenue)", tickformat=".0%", range=[0, 1])
fig.update_yaxes(title="Cumulative share of revenue", tickformat=".0%", range=[0, 1.02])
fig.update_layout(showlegend=False)
show_fig(fig, "3_revenue_concentration", "Revenue concentration", height=420)

takeaway(f"The top 20% of customers bring <b>{top20:.0%}</b> of net revenue, and half of all revenue comes from the top <b>{half:.0%}</b>. "
         + ("Revenue is highly concentrated — retaining the top tier is the priority." if top20 >= 0.7 else
            "Revenue is fairly spread — growth can come from lifting the broad middle."))

#### 📊 Chart 4 — Distributions of the R, F and M inputs
**What it shows:** the raw values behind the scores — days since last purchase, number of purchases, and net revenue per customer (log scale).
**How to read it:** use it to sanity-check the scoring. A recency histogram with peaks around campaigns or seasons explains why R cut-offs land where they do. If almost everyone has 1–2 purchases, the fixed F bins are doing their job; long purchase cycles may call for a longer lookback window in 1a.

In [20]:
# @title Chart 4 · Distributions
need("DONE_kpi", step="📈 4.3 Segment KPIs")
fig = make_subplots(rows=1, cols=3, subplot_titles=("Days since last purchase", "Purchases in window (15 = 15+)",
                                                    f"Net revenue per customer ({REPORTING_CURRENCY}, log)"))
fig.add_histogram(x=scored.recency_days, nbinsx=48, marker_color=C["blue"], row=1, col=1,
                  hovertemplate="%{x} days<br>%{y:,} customers<extra></extra>")
fv = scored.frequency.clip(upper=15)
fig.add_histogram(x=fv, xbins=dict(start=0.5, end=15.5, size=1), marker_color=C["blue"], row=1, col=2,
                  hovertemplate="%{x} purchases<br>%{y:,} customers<extra></extra>")
pos_m = scored.monetary[scored.monetary > 0]
fig.add_histogram(x=np.log10(pos_m), nbinsx=50, marker_color=C["blue"], row=1, col=3,
                  hovertemplate="%{y:,} customers<extra></extra>")
fig.update_xaxes(tickvals=[1, 2, 3, 4, 5], ticktext=["10", "100", "1k", "10k", "100k"], row=1, col=3)
fig.update_layout(showlegend=False, bargap=0.05)
n_nonpos = int((scored.monetary <= 0).sum())
show_fig(fig, "4_distributions", "Distribution of R, F and M inputs", height=360)

takeaway(f"Median customer: last bought <b>{scored.recency_days.median():.0f} days</b> ago, made <b>{scored.frequency.median():.0f}</b> "
         f"purchase(s) and spent <b>{money(scored.monetary.median())}</b> net."
         + (f" {n_nonpos:,} customers returned everything (net revenue ≤ 0) and are not shown in the revenue panel." if n_nonpos else ""))

#### 🫧 Chart 5 — Segment map
**What it shows:** each segment as a bubble — how recently its customers bought (x-axis, recent on the right), how often (y-axis), and its total net revenue (bubble size).
**How to read it:** the top-right is healthy and valuable. Big bubbles drifting left are revenue at risk. This is a good single slide to explain the segmentation to a client.

In [21]:
# @title Chart 5 · Segment map
need("DONE_kpi", step="📈 4.3 Segment KPIs")
fig = go.Figure(go.Scatter(
    x=kpi.median_days_since_purchase, y=kpi.avg_purchases, mode="markers+text", text=kpi.index,
    textposition=["bottom center" if s == "Reactivated" else "top center" for s in kpi.index],
    customdata=np.c_[kpi.customers, kpi.revenue_share, kpi.revenue_per_customer],
    marker=dict(size=np.sqrt(kpi.net_revenue.clip(lower=0) / kpi.net_revenue.max()) * 70 + 8, color=C["blue"], opacity=0.75,
                line=dict(color="white", width=2)),
    hovertemplate="<b>%{text}</b><br>Median days since purchase: %{x:.0f}<br>Avg purchases: %{y:.1f}"
                  "<br>Customers: %{customdata[0]:,.0f}<br>Revenue share: %{customdata[1]:.1%}"
                  "<br>Revenue / customer: %{customdata[2]:,.0f}<extra></extra>"))
fig.update_xaxes(title="Median days since last purchase (recent →)", autorange="reversed")
fig.update_yaxes(title="Average purchases in window", rangemode="tozero")
show_fig(fig, "5_segment_map", "Segment map (bubble size = net revenue)", height=480)

### 4.5 Key insights
**What you get:** plain-language findings generated from the numbers above — a starting point for the client summary. Always sense-check them against the charts before sharing.

In [22]:
# @title 💡 4.5 Key insights
need("DONE_kpi", step="📈 4.3 Segment KPIs")

def seg(name, col):
    return kpi.at[name, col] if name in kpi.index else np.nan
if "top20" not in globals():
    _s = scored.monetary.clip(lower=0).sort_values(ascending=False).values
    top20 = (np.cumsum(_s) / _s.sum())[max(int(np.ceil(len(_s) * 0.2)) - 1, 0)]

ins = []
ins.append(f"<b>{tot_c:,} customers</b> generated <b>{money(tot_m)}</b> net revenue in the last {LOOKBACK_MONTHS} months; "
           f"on average {scored.frequency.mean():.1f} purchases and {money(scored.monetary.mean())} per customer.")
ins.append(f"<b>Concentration:</b> the top 20% of customers account for <b>{top20:.0%}</b> of revenue.")
ins.append(f"<b>Repeat rate:</b> {repeat:.0%} of customers bought more than once in the window.")
if "Champions" in kpi.index:
    ins.append(f"<b>Champions</b> are {seg('Champions','customer_share'):.0%} of customers but {seg('Champions','revenue_share'):.0%} of revenue, "
               f"spending {seg('Champions','revenue_per_customer') / scored.monetary.mean():.1f}× the average customer.")
hv = [s for s in ["Champions", "Loyal High Value"] if s in kpi.index]
if hv:
    ins.append(f"<b>High-value core</b> ({' + '.join(hv)}): {kpi.loc[hv,'customer_share'].sum():.0%} of customers, "
               f"{kpi.loc[hv,'revenue_share'].sum():.0%} of revenue.")
if "At Risk High Value" in kpi.index:
    ins.append(f"<b>At Risk High Value:</b> {seg('At Risk High Value','customers'):,.0f} customers who produced "
               f"{money(seg('At Risk High Value','net_revenue'))} ({seg('At Risk High Value','revenue_share'):.0%} of revenue) have not bought for a median of "
               f"{seg('At Risk High Value','median_days_since_purchase'):.0f} days — the main win-back target.")
if "New Customers" in kpi.index:
    ins.append(f"<b>New Customers:</b> {seg('New Customers','customers'):,.0f} ({seg('New Customers','customer_share'):.0%}) "
               f"made their first-ever purchase in the last ~{TH['recency_days_edges'][1]:.0f} days; converting them to a "
               f"second order is the cheapest growth lever.")
if "Reactivated" in kpi.index:
    ins.append(f"<b>Reactivated:</b> {seg('Reactivated','customers'):,.0f} recent single-order customers had bought before the "
               f"window — without the customers table they would have been counted as new.")
if "Hibernating" in kpi.index:
    ins.append(f"<b>Hibernating:</b> {seg('Hibernating','customer_share'):.0%} of customers but only "
               f"{seg('Hibernating','revenue_share'):.0%} of revenue; consider low-cost reactivation or suppression.")
rr = kpi.return_rate.dropna()
if len(rr):
    ins.append(f"<b>Returns:</b> overall return rate {total.return_rate.iat[0]:.1%}; highest in <b>{esc(rr.idxmax())}</b> ({rr.max():.1%}).")
if "consent_share" in kpi:
    act = scored.marketing_consent.fillna(False).astype(bool)
    ins.append(f"<b>Activation:</b> {act.mean():.0%} of scored customers have marketing consent "
               f"({act.sum():,} addressable via email / Customer Match / Meta).")
if cust.is_outlier.any():
    ins.append(f"<b>Outliers:</b> {cust.is_outlier.sum()} customer(s) flagged as outlier/B2B "
               f"({'excluded' if OUTLIER_ACTION == 'exclude' else 'kept'}), {money(outliers.monetary.sum())} of revenue.")
if "Retail purchases linked" in coverage:
    ins.append(f"<b>Coverage caveat:</b> {coverage['Retail purchases linked']:.0%} of retail purchases are linked to a customer; "
               f"retail behaviour of anonymous shoppers is not reflected.")
if n_crit:
    ins.append("<b>⚠️ Data caveat:</b> the L0 check found critical issues — treat these numbers as indicative.")
show(card("info", f"💡 Key insights — {esc(CLIENT_NAME)}, snapshot {SNAP.date()}", "<ul>" + "".join(f"<li>{s}</li>" for s in ins) + "</ul>"))

# 5 · Export
**What you get:** all results saved to the output folder (and optionally BigQuery or a ZIP download):

| File | Contents | Use it for |
|---|---|---|
| `customer_segments.csv` | One row per customer: R/F/M values and scores, segment, consent, snapshot date | Audiences (Customer Match, Meta, CRM), Looker Studio, tracking movement between runs |
| `segment_kpis.csv` | The KPI table from 4.3 | Client deck, dashboards |
| `segment_definitions.csv` | Rules, descriptions, suggested actions | Documentation |
| `l0_dq_report.csv` · `l0_cleaning_log.csv` | Data-quality findings and row counts per cleaning step | Sending to the client |
| `rfm_thresholds.json` | The cut-offs used | Reuse in **1b · Frozen thresholds** next run |
| `run_metadata.json` | Settings, coverage and segment rules of this run | Reproducibility |
| `charts/*.html` | Interactive charts | Sharing without the notebook |

In [1]:
# @title 💾 5 · Save the results
need("DONE_kpi", step="📈 4.3 Segment KPIs")

out_dir = OUTPUT_FOLDER.strip()
if not out_dir:
    base = DATA_FOLDER.strip() if SRC in ("drive", "local") else ("/content" if IN_COLAB else ".")
    out_dir = os.path.join(base, "output", f"rfm_{SNAP.date()}")
os.makedirs(out_dir, exist_ok=True)

out = scored.reset_index()[["customer_key", "recency_days", "frequency", "monetary", "gross_revenue", "returns", "aov",
                            "R", "F", "M", "segment", "first_purchase_date", "has_prior_history",
                            "loyalty_member", "marketing_consent"]].copy()
out["rfm_code"] = out.R.astype(str) + out.F.astype(str) + out.M.astype(str)
out["snapshot_date"] = str(SNAP.date())
out["client"] = CLIENT_NAME
if OUTLIER_ACTION == "exclude" and len(outliers):
    o = outliers.reset_index()[["customer_key", "recency_days", "frequency", "monetary", "gross_revenue", "returns", "aov"]]
    o["segment"] = "Outlier (excluded)"; o["snapshot_date"] = str(SNAP.date()); o["client"] = CLIENT_NAME
    out = pd.concat([out, o], ignore_index=True)

files_out = {
    "customer_segments.csv": out,
    "segment_kpis.csv": kpi_all.rename_axis("segment").reset_index(),
    "segment_definitions.csv": seg_def,
    "l0_dq_report.csv": dq_df,
    "l0_cleaning_log.csv": clog_df,
}
for fn, df in files_out.items():
    df.to_csv(os.path.join(out_dir, fn), index=False)
with open(os.path.join(out_dir, "rfm_thresholds.json"), "w") as f:
    json.dump(TH, f, indent=2)
meta = dict(client=CLIENT_NAME, run_at=datetime.now().isoformat(timespec="seconds"), source=SOURCE,
            snapshot_date=str(SNAP.date()), window_start=str(WSTART.date()), lookback_months=LOOKBACK_MONTHS,
            reporting_currency=REPORTING_CURRENCY, outlier_action=OUTLIER_ACTION, frozen_thresholds=FROZEN,
            customers_scored=int(tot_c), segment_rules=SEGMENT_RULES,
            coverage={k: round(float(v), 4) for k, v in coverage.items()})
with open(os.path.join(out_dir, "run_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
if SAVE_CHARTS_AS_HTML and FIGS:
    os.makedirs(os.path.join(out_dir, "charts"), exist_ok=True)
    for key, fig in FIGS.items():
        fig.write_html(os.path.join(out_dir, "charts", f"{key}.html"), include_plotlyjs="cdn")

parts = []
if WRITE_TO_BIGQUERY:
    from google.cloud import bigquery
    if IN_COLAB:
        from google.colab import auth; auth.authenticate_user()
    bq = bigquery.Client(project=BQ_BILLING_PROJECT)
    job = bq.load_table_from_dataframe(out.astype({"first_purchase_date": "datetime64[ns]"}), BQ_OUTPUT_TABLE,
                                       job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"))
    job.result()
    parts.append(card("ok", "✅ Appended to BigQuery", f"<code>{esc(BQ_OUTPUT_TABLE)}</code> — {len(out):,} rows (snapshot {SNAP.date()})"))

listing = []
for root, _, fns in os.walk(out_dir):
    for fn in sorted(fns):
        pth = os.path.join(root, fn)
        listing.append(dict(File=os.path.relpath(pth, out_dir), Size=(f"{os.path.getsize(pth) / 1024:,.0f} KB" if os.path.getsize(pth) >= 1024 else "< 1 KB")))
parts.insert(0, card("ok", "✅ Results saved", f"Folder: <code>{esc(out_dir)}</code>"))
parts.append(table(pd.DataFrame(listing)))
if SAVE_CHARTS_AS_HTML and FIGS:
    parts.append('<div class="muted">Charts are saved as interactive HTML (they need internet to load the chart library).</div>')
show(*parts)

if DOWNLOAD_RESULTS_AS_ZIP:
    zpath = os.path.join("/content" if IN_COLAB else ".", f"rfm_{re.sub(r'[^A-Za-z0-9]+', '_', CLIENT_NAME)}_{SNAP.date()}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, fns in os.walk(out_dir):
            for fn in fns:
                z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), out_dir))
    if IN_COLAB:
        from google.colab import files
        files.download(zpath)
    show(card("ok", "📦 ZIP ready", f"<code>{esc(zpath)}</code>" + (" — the download starts in your browser." if IN_COLAB else "")))

NameError: name 'need' is not defined